# 01 · From Variables to a Database

**Module 08 · Persistent State & Databases**

So far the shop lived in **Python variables**: `ORDERS`, `REFUNDS`, `PENDING_APPROVALS`. That is fine for learning. But stop the program, and the desk forgets every refund it issued, every approval that was waiting, and every customer it ever talked to.

A real desk needs **persistent state**: facts that survive a restart. In this module we give the desk a real database (**SQLite**, one file, nothing to install).

| Notebook | What you will learn |
|---|---|
| 1 | **From variables to a database** (this one): why variables are not enough, and SQL in five minutes |
| 2 | **Money must be safe**: transactions, UNIQUE rules, and a real race condition |
| 3 | **A desk that remembers**: customer history, and how memory can be poisoned |
| 4 | **Restart, crash, grow up**: approvals that survive, a power cut, backups, and when SQLite is not enough |

> This notebook uses no OpenAI calls, so it costs nothing.

## 1. What a restart does to Python variables
Two separate runs of the same small program. The first one issues a refund, so the second one should know about it:

In [1]:
import subprocess
import sys

program = """
REFUNDS = []                                   # the shop's memory: a Python variable
print("Refunds known at the start:", len(REFUNDS))
REFUNDS.append("A1001")                        # issue a refund
print("Refunds known at the end  :", len(REFUNDS))
"""

for run in [1, 2]:
    print("Run", run)
    result = subprocess.run([sys.executable, "-c", program], capture_output=True, text=True)
    print(result.stdout)

Run 1
Refunds known at the start: 0
Refunds known at the end  : 1

Run 2


Refunds known at the start: 0
Refunds known at the end  : 1



Run 2 starts with **0 refunds**. The refund from run 1 is gone. For a shop that means it could refund the same order again, forget a customer's complaint, and lose every approval that was waiting.

## 2. "I will just save it in a JSON file"
It is the first idea everyone has, and it works... until two things happen at the same time. Here two programs, **A** and **B**, both read the file, add one refund each, and write it back:

In [2]:
import json
import tempfile
from pathlib import Path

scratch = Path(tempfile.mkdtemp())          # a throwaway folder for this notebook
refunds_file = scratch / "refunds.json"
refunds_file.write_text("[]")

# A and B both read the file at the same moment...
a_sees = json.loads(refunds_file.read_text())
b_sees = json.loads(refunds_file.read_text())

# ...each adds its own refund...
a_sees.append("A1001")
b_sees.append("A1002")

# ...and each writes the whole file back.
refunds_file.write_text(json.dumps(a_sees))
refunds_file.write_text(json.dumps(b_sees))

print("The file now says:", refunds_file.read_text())
print("A's refund (A1001) is gone. Nobody got an error.")

The file now says: ["A1002"]
A's refund (A1001) is gone. Nobody got an error.


This is called a **lost update**. A file has no idea that two programs are using it. A database does: it takes care of **locks, transactions and crashes** for you.

| | Python variable | JSON file | Database (SQLite) |
|---|---|---|---|
| Survives a restart | no | yes | yes |
| Two writers at once | n/a | **loses data** | safe |
| Crash in the middle of a write | everything lost | **file can be half-written** | all or nothing |
| Ask "all refunds over $100 last week" | loop by hand | loop by hand | one line of SQL |
| Rules enforced for you (no duplicate refund) | no | no | **yes** (UNIQUE, CHECK) |

## 3. SQLite and SQL in five minutes
**SQLite** is a database that is a single file. Python includes it (`import sqlite3`). **SQL** is the language you use to talk to any database. Four statements are enough for this module:

| SQL | Meaning |
|---|---|
| `CREATE TABLE` | make a table (like a sheet with fixed columns) |
| `INSERT` | add a row |
| `SELECT ... WHERE` | find rows |
| `UPDATE` | change rows |

In [3]:
import sqlite3

demo_db = scratch / "demo.db"
conn = sqlite3.connect(demo_db)

conn.execute("CREATE TABLE orders (order_id TEXT PRIMARY KEY, item TEXT, price REAL)")
conn.execute("INSERT INTO orders VALUES ('A1001', 'Wireless headphones', 79.0)")
conn.execute("INSERT INTO orders VALUES ('A1002', 'Running shoes', 120.0)")
conn.execute("INSERT INTO orders VALUES ('A1004', '4K television', 250.0)")
conn.commit()          # until we commit, the rows are not saved

print("Orders under $200:")
for row in conn.execute("SELECT order_id, item, price FROM orders WHERE price < ?", (200,)):
    print("  ", row)

Orders under $200:
   ('A1001', 'Wireless headphones', 79.0)
   ('A1002', 'Running shoes', 120.0)


Notice the `?` in the query. We did not paste the value into the text. That is not a style choice, it is **security**.

## 4. The question mark that stops SQL injection
If you build the query by gluing text together, a customer can write SQL for you. Compare the two ways, with a "customer" who types an order id like this: `x' OR '1'='1`

In [4]:
order_id = "x' OR '1'='1"      # what a malicious customer types

# WRONG: glue the text into the query. The customer's text becomes part of the SQL.
query = "SELECT item FROM orders WHERE order_id = '" + order_id + "'"
print("The query we ran:", query)
print("Rows returned   :", conn.execute(query).fetchall(), "  <- every order in the shop!")

# RIGHT: a placeholder. The value is data, never SQL.
print("With a ?         :", conn.execute("SELECT item FROM orders WHERE order_id = ?", (order_id,)).fetchall())

The query we ran: SELECT item FROM orders WHERE order_id = 'x' OR '1'='1'
Rows returned   : [('Wireless headphones',), ('Running shoes',), ('4K television',)]   <- every order in the shop!
With a ?         : []


> **Rule: never build SQL by joining strings with anything that came from a user** (or from a model!). Always use `?`. Every function in `db.py` does.
>
> This matters for agents twice over: the order id that reaches your tool was typed by a customer and then repeated by a model.

## 5. It survives a restart
Close our connection, and read the same file from a **different program**:

In [5]:
conn.close()

reader = """
import sqlite3, sys
conn = sqlite3.connect(sys.argv[1])
print("A different process sees:", conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0], "orders")
"""
print(subprocess.run([sys.executable, "-c", reader, str(demo_db)], capture_output=True, text=True).stdout)

A different process sees: 3 orders



## 6. The desk's database
The module's `db.py` creates five tables. Read the SQL once, because everything else in the module rests on it:

| Table | Holds | A rule the database enforces |
|---|---|---|
| `orders` | the shop's orders | `order_id` is the primary key |
| `refunds` | refunds that were issued | **`order_id` is UNIQUE**: one refund per order, ever |
| `approvals` | big refunds waiting for a person | **only one `pending` row per order** |
| `tickets` | every email and reply | |
| `audit` | who did what, when | |

In [6]:
import db

db.use_database(scratch / "desk.db")     # a throwaway database, so your real desk.db is untouched
db.reset_db()

conn = db.connect()
for row in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
    print("table:", row["name"])
conn.close()

print()
print(db.SCHEMA[:900], "...")

table: approvals
table: audit
table: orders
table: refunds
table: sqlite_sequence
table: tickets


CREATE TABLE IF NOT EXISTS orders (
    order_id          TEXT PRIMARY KEY,
    customer_id       TEXT NOT NULL,
    item              TEXT NOT NULL,
    price             REAL NOT NULL,
    status            TEXT NOT NULL,
    delivered_on      TEXT,
    expected_delivery TEXT
);

-- UNIQUE on order_id: the database itself refuses a second refund for the same order.
CREATE TABLE IF NOT EXISTS refunds (
    refund_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id    TEXT NOT NULL UNIQUE REFERENCES orders(order_id),
    amount      REAL NOT NULL CHECK (amount > 0),
    approved_by TEXT NOT NULL,
    created_at  TEXT NOT NULL
);

-- Refunds above the automatic limit wait here until a person decides.
CREATE TABLE IF NOT EXISTS approvals (
    approval_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id     TEXT NOT NULL REFERENCES orders(order_id),
    amount       REAL NOT NULL,
    st .

**Next:** the tables hold money decisions. Notebook 2 shows how to keep them correct when things happen at the same time, or fail in the middle.